# LeetCode 636 — Exclusive Time of Functions bằng Python

Link đề: https://leetcode.com/problems/exclusive-time-of-functions/ · Độ khó: **Medium** · Chủ đề: Stack, Array

Skeleton LeetCode:

```python
class Solution:
    def exclusiveTime(self, n: int, logs: List[str]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một CPU **đơn luồng** chạy các hàm lồng nhau (có thể **đệ quy**); từ các log `"id:start|end:timestamp"` tính **thời gian độc quyền** (exclusive time) của từng hàm — thời gian CPU **thực sự chạy mã của hàm đó**, không tính phần chạy hàm con.
2. Nắm các công cụ Python: `str.split(":")` + `int()`, dùng `list` làm **ngăn xếp** (`append`, `pop`, `[-1]`), khoảng thời gian **đóng** `[start, end]` có độ dài `end - start + 1`, đệ quy với con trỏ chung (closure), và cách mô hình hóa "đầu đơn vị thời gian / cuối đơn vị thời gian".
3. Đi từ **mô phỏng từng đơn vị thời gian** → **ngăn xếp + mốc `pre`** → **ngăn xếp khung gọi: thời gian tổng trừ thời gian của hàm con** → **đệ quy theo cấu trúc lồng nhau** của log.
4. Hiểu mấu chốt: tại mọi thời điểm, hàm đang chạy là **đỉnh ngăn xếp**; chỉ cần tính **chênh lệch giữa hai sự kiện liên tiếp** và cộng cho đúng hàm đang ở đỉnh, nên không bao giờ phải đi từng đơn vị thời gian (timestamp tới `10^9`).
5. Tránh các lỗi kinh điển: quên `+ 1` ở sự kiện `end`, đặt lại mốc `pre = cur` (thay vì `cur + 1`) sau `end`, quên cộng thời gian cho hàm cha khi hàm con bắt đầu, đọc timestamp thành chuỗi, và cộng thời gian tổng mà không trừ thời gian hàm con (đếm trùng với đệ quy).

## 2. Đề bài & ví dụ

CPU **đơn luồng** chạy một chương trình gồm `n` hàm, mỗi hàm có `ID` duy nhất từ `0` đến `n - 1`.

Các lời gọi hàm lưu trong **ngăn xếp gọi** (call stack): khi một lời gọi bắt đầu, ID của nó được đẩy lên ngăn xếp; khi kết thúc, ID bị lấy ra. Hàm ở **đỉnh** ngăn xếp là hàm **đang chạy**. Mỗi lần một hàm bắt đầu hoặc kết thúc, ta ghi một log gồm ID, loại sự kiện (bắt đầu/kết thúc) và timestamp.

Cho danh sách `logs`, `logs[i]` có dạng chuỗi `"{function_id}:{"start" | "end"}:{timestamp}"`. Ví dụ `"0:start:3"`: hàm 0 bắt đầu **ở đầu** đơn vị thời gian 3; `"1:end:2"`: hàm 1 kết thúc **ở cuối** đơn vị thời gian 2. Một hàm có thể được gọi **nhiều lần, kể cả đệ quy**.

**Thời gian độc quyền** của một hàm là **tổng** thời gian thực thi của mọi lời gọi hàm đó (không tính thời gian của hàm khác mà nó gọi). Hãy trả về mảng thời gian độc quyền, phần tử thứ `i` ứng với hàm `i`.

**Ví dụ 1:** `n = 2`, `logs = ["0:start:0","1:start:2","1:end:5","0:end:6"]` → `[3, 4]`.
Hàm 0 chạy đơn vị 0 và 1 (2 đơn vị), nhường cho hàm 1 chạy các đơn vị 2..5 (4 đơn vị), rồi chạy lại đơn vị 6 (1 đơn vị): `2 + 1 = 3`.

**Ví dụ 2:** `n = 1`, `logs = ["0:start:0","0:start:2","0:end:5","0:start:6","0:end:6","0:end:7"]` → `[8]` (đệ quy: `2 + 4 + 1 + 1`).

**Ví dụ 3:** `n = 2`, `logs = ["0:start:0","0:start:2","0:end:5","1:start:6","1:end:6","0:end:7"]` → `[7, 1]`.

**Ràng buộc:** `1 <= n <= 100`; `2 <= logs.length <= 500`; `0 <= function_id < n`; `0 <= timestamp <= 10^9`; không có hai sự kiện `start` cùng timestamp; không có hai sự kiện `end` cùng timestamp; mỗi `start` có một `end` tương ứng.

In [1]:
import random
import time
from typing import List

EX = [
    (2, ["0:start:0", "1:start:2", "1:end:5", "0:end:6"], [3, 4]),
    (1, ["0:start:0", "0:start:2", "0:end:5", "0:start:6", "0:end:6", "0:end:7"], [8]),
    (2, ["0:start:0", "0:start:2", "0:end:5", "1:start:6", "1:end:6", "0:end:7"], [7, 1]),
]
for n, logs, want in EX:
    print(n, logs, "->", want)

2 ['0:start:0', '1:start:2', '1:end:5', '0:end:6'] -> [3, 4]
1 ['0:start:0', '0:start:2', '0:end:5', '0:start:6', '0:end:6', '0:end:7'] -> [8]
2 ['0:start:0', '0:start:2', '0:end:5', '1:start:6', '1:end:6', '0:end:7'] -> [7, 1]


## 3. Kiến thức nền

### 3.1. Tách chuỗi log: `split` và `int`

`"1:end:5".split(":")` cho `["1", "end", "5"]` — ba **chuỗi**. ID và timestamp phải đổi sang số bằng `int(...)` trước khi cộng trừ (nếu quên, `"5" - "2"` báo lỗi).

In [2]:
log = "1:end:5"
parts = log.split(":")
print(parts)
i, op, t = parts
i, t = int(i), int(t)
print(i, op, t, "| op bắt đầu bằng s?", op[0] == "s", op == "start")
try:
    "5" - "2"
except TypeError as e:
    print("TypeError:", e)

['1', 'end', '5']
1 end 5 | op bắt đầu bằng s? False False
TypeError: unsupported operand type(s) for -: 'str' and 'str'


### 3.2. `list` làm ngăn xếp (stack)

Ngăn xếp: **vào sau, ra trước** (LIFO). Trong Python: `stk.append(x)` đẩy lên đỉnh, `stk.pop()` lấy đỉnh ra, `stk[-1]` xem đỉnh **không lấy ra**, `if stk:` kiểm tra không rỗng. Mọi thao tác `O(1)`.

In [3]:
stk = []
stk.append(0)
stk.append(1)
stk.append(1)
print(stk, "đỉnh =", stk[-1])
print("pop ->", stk.pop(), stk)
print("pop ->", stk.pop(), stk, "| còn phần tử?", bool(stk))
stk.pop()
print(stk, "| rỗng?", not stk)
try:
    stk.pop()
except IndexError as e:
    print("IndexError:", e)

[0, 1, 1] đỉnh = 1
pop -> 1 [0, 1]
pop -> 1 [0] | còn phần tử? True
[] | rỗng? True
IndexError: pop from empty list


### 3.3. Ngăn xếp gọi hàm và "ai đang chạy?"

Ngăn xếp gọi (call stack): khi `f` gọi `g`, `g` được đẩy lên trên `f`; `g` chạy cho đến khi kết thúc rồi bị lấy ra, `f` **tiếp tục** từ chỗ đã dừng. Ở mọi lúc, **đỉnh ngăn xếp** là hàm **đang chiếm CPU**. Hàm cùng id có thể xuất hiện **nhiều lần** trong ngăn xếp (đệ quy).

In [4]:
def show_stack_over_time(events):
    stack = []
    for who, op in events:
        if op == "start":
            stack.append(who)
        else:
            assert stack.pop() == who
        print(f"{who} {op:<5} -> ngăn xếp {stack}  | đang chạy: {stack[-1] if stack else '(trống)'}")


show_stack_over_time([(0, "start"), (1, "start"), (1, "end"), (0, "start"), (0, "end"), (0, "end")])

0 start -> ngăn xếp [0]  | đang chạy: 0
1 start -> ngăn xếp [0, 1]  | đang chạy: 1
1 end   -> ngăn xếp [0]  | đang chạy: 0
0 start -> ngăn xếp [0, 0]  | đang chạy: 0
0 end   -> ngăn xếp [0]  | đang chạy: 0
0 end   -> ngăn xếp []  | đang chạy: (trống)


### 3.4. Đơn vị thời gian và khoảng **đóng** `[start, end]`

Thời gian chia thành các **ô** (đơn vị) `0, 1, 2, ...`. `start t` nghĩa là hàm bắt đầu ở **đầu ô `t`** ⇒ ô `t` thuộc về hàm. `end t` nghĩa là hàm kết thúc ở **cuối ô `t`** ⇒ ô `t` **vẫn thuộc về hàm**. Do đó hàm chạy các ô `start, start+1, ..., end` — gồm **`end - start + 1`** ô (khoảng **đóng**). Ví dụ `start 2, end 5` chiếm 4 ô: `2, 3, 4, 5`.

In [5]:
s, e = 2, 5
cells = list(range(s, e + 1))
print(cells, "-> độ dài", len(cells), "==", e - s + 1)
print("hàm cha ngừng trước ô", s, "và chạy tiếp từ ô", e + 1)

[2, 3, 4, 5] -> độ dài 4 == 4
hàm cha ngừng trước ô 2 và chạy tiếp từ ô 6


### 3.5. Đệ quy và con trỏ chia sẻ (closure)

Hàm lồng nhau có thể đọc/ghi biến của hàm ngoài nếu dùng `nonlocal` hoặc bọc trong `list` (`pos = [0]`, dùng `pos[0] += 1`). Cách này dùng để một chuỗi lời gọi đệ quy cùng đọc theo một "con trỏ" trên mảng log.

In [6]:
def demo_consume(items):
    pos = [0]                                     # con trỏ chia sẻ

    def take_group():
        # lấy một "ngoặc" '(' ... ')' đệ quy, trả về độ sâu tối đa
        assert items[pos[0]] == "("
        pos[0] += 1
        depth = 0
        while items[pos[0]] == "(":
            depth = max(depth, take_group())
        assert items[pos[0]] == ")"
        pos[0] += 1
        return depth + 1

    out = []
    while pos[0] < len(items):
        out.append(take_group())
    return out


print(demo_consume(list("(()())()")), demo_consume(list("((()))")))

[2, 1] [3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy **Ví dụ 1**: `["0:start:0", "1:start:2", "1:end:5", "0:end:6"]`. Vẽ **dòng thời gian** các ô:

```
ô:         0   1   2   3   4   5   6
chạy:      f0  f0  f1  f1  f1  f1  f0
```

- Hàm 0 bắt đầu ở đầu ô 0 → chiếm ô 0, 1; ở đầu ô 2 hàm 1 xen vào (nên f0 chỉ có ô 0 và 1).
- Hàm 1 chạy các ô 2, 3, 4, 5 và kết thúc ở cuối ô 5.
- Hàm 0 tiếp tục từ ô 6 (ô **ngay sau** ô kết thúc của hàm 1) và kết thúc ở cuối ô 6.

Đếm: `f0` có ô `0, 1, 6` → 3; `f1` có ô `2..5` → 4. Kết quả `[3, 4]`.

**Quan sát then chốt:** ta không cần đi từng ô. Mỗi **sự kiện** chỉ làm thay đổi "ai đang chạy", nên chỉ cần biết **khoảng cách giữa hai sự kiện liên tiếp** và **ai đang đứng đỉnh** trong khoảng đó:

- trước `1:start:2`, đỉnh là f0 từ mốc `0` đến `2`: cộng `2 - 0 = 2` cho f0;
- giữa `1:start:2` và `1:end:5`, đỉnh là f1 từ ô `2` đến ô `5` (gồm cả hai đầu): cộng `5 - 2 + 1 = 4` cho f1;
- sau `1:end:5`, f0 chạy tiếp từ ô `6` đến ô `6`: cộng `6 - 6 + 1 = 1` cho f0.

## 5. Cách 1 — Brute Force / baseline: mô phỏng **từng đơn vị thời gian**

**Ý tưởng:** đi qua từng ô thời gian `t` từ ô đầu tiên đến ô cuối cùng. Ở mỗi ô:

1. Xử lý mọi sự kiện **`start` tại `t`** (đẩy lên ngăn xếp) — vì `start` xảy ra ở **đầu** ô.
2. Nếu ngăn xếp không rỗng: hàm ở **đỉnh** chạy ô này ⇒ `ans[đỉnh] += 1`.
3. Xử lý mọi sự kiện **`end` tại `t`** (lấy khỏi ngăn xếp) — vì `end` xảy ra ở **cuối** ô.

Log được cho theo **thứ tự thời gian** (và trong cùng một timestamp thì `start` đứng trước `end`), nên dùng một con trỏ `p` đi dần qua log. Đây là bản dịch nguyên văn của đề thành mô phỏng, cũng là **oracle** (ở mức đơn vị) để kiểm tra các cách sau trên timestamp nhỏ.

In [7]:
def parse(log):
    i, op, t = log.split(":")
    return int(i), op, int(t)


def exclusive_unit(n, logs):
    ev = [parse(l) for l in logs]
    ans = [0] * n
    stk = []
    p = 0
    t = ev[0][2]
    last = max(e[2] for e in ev)
    while t <= last:
        while p < len(ev) and ev[p][1] == "start" and ev[p][2] == t:
            stk.append(ev[p][0])                  # start ở ĐẦU ô t
            p += 1
        if stk:
            ans[stk[-1]] += 1                     # đỉnh chạy ô t
        while p < len(ev) and ev[p][1] == "end" and ev[p][2] == t:
            stk.pop()                             # end ở CUỐI ô t
            p += 1
        t += 1
    return ans


for n, logs, want in EX:
    got = exclusive_unit(n, logs)
    print(got, want)
    assert got == want

[3, 4] [3, 4]
[8] [8]
[7, 1] [7, 1]


**Độ phức tạp:** thời gian **`O(T + L)`** với `T = (timestamp cuối − timestamp đầu + 1)` — **tới `10^9`** theo đề; `L` là số log. Space `O(L)`.

**Nút thắt:** với `timestamp` lớn (`10^9`) vòng lặp theo ô **không thể chạy kịp**. Nhưng giữa hai sự kiện liên tiếp, **không có gì thay đổi**: hàm đỉnh ngăn xếp chạy suốt cả đoạn. Ta chỉ cần cộng cả **độ dài đoạn** một lần: `ans[đỉnh] += độ dài`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 — chỉ cần xét các sự kiện.** Giữa hai sự kiện liên tiếp, ngăn xếp không đổi nên đỉnh không đổi; mọi ô trong đoạn đó thuộc về cùng một hàm. Vậy thay vì cộng 1 cho mỗi ô, cộng **độ dài đoạn** cho đỉnh.

**Bổ đề 2 — quy ước mốc.** Gọi `pre` là **ô đầu tiên chưa được tính** (mốc "đang tính dở"):

- Khi gặp **`start` ở `cur`**: các ô `pre, ..., cur − 1` (nếu có) thuộc về **hàm đỉnh hiện tại** (hàm cha) ⇒ `ans[đỉnh] += cur − pre`; rồi đẩy hàm mới và đặt `pre = cur` (hàm mới chiếm từ ô `cur`).
- Khi gặp **`end` ở `cur`**: các ô `pre, ..., cur` (**gồm** ô `cur`) thuộc về hàm đỉnh ⇒ `ans[đỉnh] += cur − pre + 1`; lấy hàm đó ra; đặt `pre = cur + 1` (hàm cha — nếu còn — chạy tiếp từ ô sau).

Hai quy ước `cur − pre` (đầu ô, nửa mở) và `cur − pre + 1` (cuối ô, đóng) chính là chỗ người mới hay nhầm nhất.

**Bổ đề 3 — hai cách nhìn tương đương.** Ta có thể tính theo **tổng thời gian của từng lời gọi** `end − start + 1` rồi **trừ** tổng thời gian của các lời gọi con trực tiếp: `exclusive = inclusive − Σ inclusive(con)`. Kết hợp với cấu trúc **lồng nhau** (như ngoặc đơn), ta có cả cách dùng **ngăn xếp khung gọi** lẫn cách **đệ quy**.

Từ đó có các cách: **ngăn xếp + mốc `pre`** (Cách 2), **ngăn xếp khung: tổng − con** (Cách 3), **đệ quy** (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (mô phỏng từng ô, mục 5) và ba cách dưới đây. Cách 2 tính **theo khoảng giữa hai sự kiện**; Cách 3 tính **theo từng lời gọi** (tổng thời gian trừ phần của con); Cách 4 dùng chính **ngăn xếp của ngôn ngữ** (đệ quy) thay cho ngăn xếp tự dựng.

### Cách 2 — Ngăn xếp + mốc `pre` (cách chính thức)

**Giải thích từng đoạn:**

1. `stk` chứa ID các hàm đang chạy (đỉnh = hàm đang chạy); `ans[i]` là kết quả; `pre` là mốc ô đầu tiên chưa tính.
2. Với mỗi log, tách ra `i, op, cur`.
3. `start`: nếu `stk` không rỗng thì `ans[stk[-1]] += cur - pre`; sau đó `stk.append(i)` và `pre = cur`.
4. `end`: `ans[stk.pop()] += cur - pre + 1`; rồi `pre = cur + 1`.

In [8]:
def exclusive_stack(n, logs):
    stk = []
    ans = [0] * n
    pre = 0
    for log in logs:
        i, op, t = log.split(":")
        i, cur = int(i), int(t)
        if op[0] == "s":                              # start
            if stk:
                ans[stk[-1]] += cur - pre             # phần hàm cha đã chạy trước khi con bắt đầu
            stk.append(i)
            pre = cur
        else:                                         # end
            ans[stk.pop()] += cur - pre + 1           # +1 vì ô cur vẫn thuộc về hàm vừa kết thúc
            pre = cur + 1
    return ans


for n, logs, want in EX:
    got = exclusive_stack(n, logs)
    print(got, want)
    assert got == want

[3, 4] [3, 4]
[8] [8]
[7, 1] [7, 1]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def exclusiveTime(self, n: int, logs: List[str]) -> List[int]:
        stk = []
        ans = [0] * n
        pre = 0
        for log in logs:
            i, op, t = log.split(":")
            i, cur = int(i), int(t)
            if op[0] == "s":
                if stk:
                    ans[stk[-1]] += cur - pre
                stk.append(i)
                pre = cur
            else:
                ans[stk.pop()] += cur - pre + 1
                pre = cur + 1
        return ans
```

**Độ phức tạp:** một lượt qua `L` log, mỗi log `O(1)` (tách chuỗi coi như hằng) ⇒ **`O(L)`**. Space `O(L)` (ngăn xếp sâu nhất `L/2`), cộng `O(n)` cho `ans`.

### Cách 3 — Ngăn xếp khung gọi: thời gian **tổng** trừ thời gian của **hàm con**

**Giải thích từng đoạn:**

1. Mỗi phần tử ngăn xếp là một **khung** `[id, start, child]`: id hàm, thời điểm bắt đầu, và **tổng thời gian tổng của các con trực tiếp đã kết thúc** (`child`, ban đầu `0`).
2. `start`: đẩy khung `[i, t, 0]`.
3. `end`: lấy khung ra, `total = t − start + 1` (khoảng đóng); `ans[id] += total − child` (thời gian của chính nó = tổng trừ phần con); nếu còn hàm cha ở đỉnh thì `cha.child += total`.

Ưu điểm: **không cần mốc `pre`**, mỗi lời gọi tự tính độc lập, dễ mở rộng (độ sâu, thời gian tổng, ...).

In [9]:
def exclusive_frames(n, logs):
    stk = []                                          # mỗi phần tử: [id, start, child_time]
    ans = [0] * n
    for log in logs:
        i, op, t = parse(log)
        if op == "start":
            stk.append([i, t, 0])
        else:
            fid, s, child = stk.pop()
            assert fid == i
            total = t - s + 1                         # thời gian TỔNG của lời gọi này (gồm cả con)
            ans[fid] += total - child                 # chỉ giữ phần của chính nó
            if stk:
                stk[-1][2] += total                   # báo cho cha biết con đã tốn bao nhiêu
    return ans


for n, logs, want in EX:
    got = exclusive_frames(n, logs)
    print(got, want)
    assert got == want

[3, 4] [3, 4]
[8] [8]
[7, 1] [7, 1]


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def exclusiveTime(self, n: int, logs: List[str]) -> List[int]:
        stk = []                                      # [id, start, child]
        ans = [0] * n
        for log in logs:
            i, op, t = log.split(":")
            i, t = int(i), int(t)
            if op[0] == "s":
                stk.append([i, t, 0])
            else:
                fid, s, child = stk.pop()
                total = t - s + 1
                ans[fid] += total - child
                if stk:
                    stk[-1][2] += total
        return ans
```

**Độ phức tạp:** `O(L)` thời gian, `O(L)` không gian (mỗi khung 3 số).

### Cách 4 — Đệ quy theo cấu trúc lồng nhau (ngăn xếp của ngôn ngữ)

**Giải thích từng đoạn:**

1. Log có dạng **ngoặc lồng nhau**: `start` mở, `end` đóng; giữa `start` và `end` tương ứng là các lời gọi con (cũng `start ... end`).
2. `call()` xử lý **một lời gọi**: đọc `start` (nhớ `s`), rồi **lặp khi log kế tiếp là `start`** — gọi `call()` cho con và cộng `total` của con vào `child`; cuối cùng đọc `end`, tính `total = e − s + 1`, `ans[id] += total − child`, trả về `total`.
3. Vòng ngoài gọi `call()` cho đến khi hết log (các lời gọi ở mức cao nhất).

Độ sâu đệ quy tối đa bằng độ sâu lồng nhau ≤ `L/2 = 250` — thoải mái so với giới hạn `1000` mặc định của Python.

In [10]:
def exclusive_recursive(n, logs):
    ev = [parse(l) for l in logs]
    ans = [0] * n
    pos = [0]

    def call():
        i, op, s = ev[pos[0]]
        assert op == "start"
        pos[0] += 1
        child = 0
        while ev[pos[0]][1] == "start":               # các lời gọi con trực tiếp
            child += call()
        j, op2, e = ev[pos[0]]
        assert op2 == "end" and j == i                # end khớp đúng start
        pos[0] += 1
        total = e - s + 1
        ans[i] += total - child
        return total

    while pos[0] < len(ev):
        call()
    return ans


for n, logs, want in EX:
    got = exclusive_recursive(n, logs)
    print(got, want)
    assert got == want

[3, 4] [3, 4]
[8] [8]
[7, 1] [7, 1]


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def exclusiveTime(self, n: int, logs: List[str]) -> List[int]:
        ev = []
        for log in logs:
            i, op, t = log.split(":")
            ev.append((int(i), op, int(t)))
        ans = [0] * n
        pos = 0

        def call() -> int:
            nonlocal pos
            i, _, s = ev[pos]
            pos += 1
            child = 0
            while ev[pos][1] == "start":
                child += call()
            e = ev[pos][2]
            pos += 1
            total = e - s + 1
            ans[i] += total - child
            return total

        while pos < len(ev):
            call()
        return ans
```

**Độ phức tạp:** mỗi log được đọc đúng một lần ⇒ `O(L)` thời gian; Space `O(độ sâu lồng nhau) ≤ O(L)` ngăn xếp đệ quy.

### Kiểm tra nhanh bốn cách trên ba ví dụ

In [11]:
IMPLS = {
    "unit": exclusive_unit,
    "stack": exclusive_stack,
    "frames": exclusive_frames,
    "recursive": exclusive_recursive,
}
for n, logs, want in EX:
    print([f(n, logs) for f in IMPLS.values()], want)
    assert all(f(n, logs) == want for f in IMPLS.values())

[[3, 4], [3, 4], [3, 4], [3, 4]] [3, 4]
[[8], [8], [8], [8]] [8]
[[7, 1], [7, 1], [7, 1], [7, 1]] [7, 1]


## 8. TRACE từng bước

**Trace Cách 2 trên Ví dụ 1** `["0:start:0","1:start:2","1:end:5","0:end:6"]`:

| Log | ngăn xếp trước | hành động | `ans` | `pre` | ngăn xếp sau |
|---|---|---|---|---|---|
| `0:start:0` | `[]` | rỗng ⇒ không cộng; push 0 | `[0,0]` | 0 | `[0]` |
| `1:start:2` | `[0]` | `ans[0] += 2 − 0 = 2`; push 1 | `[2,0]` | 2 | `[0,1]` |
| `1:end:5` | `[0,1]` | `ans[1] += 5 − 2 + 1 = 4`; pop | `[2,4]` | 6 | `[0]` |
| `0:end:6` | `[0]` | `ans[0] += 6 − 6 + 1 = 1`; pop | `[3,4]` | 7 | `[]` |

**Ví dụ 2 (đệ quy, hàm 0 gọi chính nó):** `["0:start:0","0:start:2","0:end:5","0:start:6","0:end:6","0:end:7"]`:

| Log | hành động | `ans[0]` | `pre` | ngăn xếp sau |
|---|---|---|---|---|
| `0:start:0` | push | 0 | 0 | `[0]` |
| `0:start:2` | cha chạy `2 − 0 = 2`; push | 2 | 2 | `[0,0]` |
| `0:end:5` | `5 − 2 + 1 = 4` | 6 | 6 | `[0]` |
| `0:start:6` | cha chạy `6 − 6 = 0`; push | 6 | 6 | `[0,0]` |
| `0:end:6` | `6 − 6 + 1 = 1` | 7 | 7 | `[0]` |
| `0:end:7` | `7 − 7 + 1 = 1` | 8 | 8 | `[]` |

Hàm `trace_stack` in đúng các bước trên:

In [12]:
def trace_stack(n, logs):
    stk, ans, pre = [], [0] * n, 0
    print(f"{'log':<12}{'ngăn xếp trước':<16}{'hành động':<34}{'ans':<12}{'pre':<5}{'ngăn xếp sau'}")
    for log in logs:
        i, op, cur = parse(log)
        before = list(stk)
        if op == "start":
            if stk:
                add = cur - pre
                act = f"ans[{stk[-1]}] += {cur}-{pre} = {add}; push {i}"
                ans[stk[-1]] += add
            else:
                act = f"rỗng, không cộng; push {i}"
            stk.append(i)
            pre = cur
        else:
            add = cur - pre + 1
            top = stk.pop()
            act = f"ans[{top}] += {cur}-{pre}+1 = {add}; pop"
            ans[top] += add
            pre = cur + 1
        print(f"{log:<12}{str(before):<16}{act:<34}{str(ans):<12}{pre:<5}{stk}")
    print("=>", ans, "\n")
    return ans


for n, logs, want in EX:
    assert trace_stack(n, logs) == want

log         ngăn xếp trước  hành động                         ans         pre  ngăn xếp sau
0:start:0   []              rỗng, không cộng; push 0          [0, 0]      0    [0]
1:start:2   [0]             ans[0] += 2-0 = 2; push 1         [2, 0]      2    [0, 1]
1:end:5     [0, 1]          ans[1] += 5-2+1 = 4; pop          [2, 4]      6    [0]
0:end:6     [0]             ans[0] += 6-6+1 = 1; pop          [3, 4]      7    []
=> [3, 4] 

log         ngăn xếp trước  hành động                         ans         pre  ngăn xếp sau
0:start:0   []              rỗng, không cộng; push 0          [0]         0    [0]
0:start:2   [0]             ans[0] += 2-0 = 2; push 0         [2]         2    [0, 0]
0:end:5     [0, 0]          ans[0] += 5-2+1 = 4; pop          [6]         6    [0]
0:start:6   [0]             ans[0] += 6-6 = 0; push 0         [6]         6    [0, 0]
0:end:6     [0, 0]          ans[0] += 6-6+1 = 1; pop          [7]         7    [0]
0:end:7     [0]             ans[0] += 7-7+1 = 1; 

**Trace Cách 3 (khung gọi)** trên Ví dụ 2: mỗi lần `end` in ra `total`, `child` và phần của chính hàm.

In [13]:
def trace_frames(n, logs):
    stk, ans = [], [0] * n
    for log in logs:
        i, op, t = parse(log)
        if op == "start":
            stk.append([i, t, 0])
            print(f"{log:<12} push khung {stk[-1]}   ngăn xếp: {stk}")
        else:
            fid, s, child = stk.pop()
            total = t - s + 1
            own = total - child
            ans[fid] += own
            if stk:
                stk[-1][2] += total
            print(f"{log:<12} pop  khung [{fid},{s},{child}]  total={t}-{s}+1={total}, child={child}, "
                  f"phần riêng={own}  ans={ans}  cha: {stk[-1] if stk else None}")
    print("=>", ans, "\n")
    return ans


assert trace_frames(*EX[1][:2]) == [8]
assert trace_frames(*EX[2][:2]) == [7, 1]

0:start:0    push khung [0, 0, 0]   ngăn xếp: [[0, 0, 0]]
0:start:2    push khung [0, 2, 0]   ngăn xếp: [[0, 0, 0], [0, 2, 0]]
0:end:5      pop  khung [0,2,0]  total=5-2+1=4, child=0, phần riêng=4  ans=[4]  cha: [0, 0, 4]
0:start:6    push khung [0, 6, 0]   ngăn xếp: [[0, 0, 4], [0, 6, 0]]
0:end:6      pop  khung [0,6,0]  total=6-6+1=1, child=0, phần riêng=1  ans=[5]  cha: [0, 0, 5]
0:end:7      pop  khung [0,0,5]  total=7-0+1=8, child=5, phần riêng=3  ans=[8]  cha: None
=> [8] 

0:start:0    push khung [0, 0, 0]   ngăn xếp: [[0, 0, 0]]
0:start:2    push khung [0, 2, 0]   ngăn xếp: [[0, 0, 0], [0, 2, 0]]
0:end:5      pop  khung [0,2,0]  total=5-2+1=4, child=0, phần riêng=4  ans=[4, 0]  cha: [0, 0, 4]
1:start:6    push khung [1, 6, 0]   ngăn xếp: [[0, 0, 4], [1, 6, 0]]
1:end:6      pop  khung [1,6,0]  total=6-6+1=1, child=0, phần riêng=1  ans=[4, 1]  cha: [0, 0, 5]
0:end:7      pop  khung [0,0,5]  total=7-0+1=8, child=5, phần riêng=3  ans=[7, 1]  cha: None
=> [7, 1] 



**Trace Cách 4 (đệ quy)** — thụt đầu dòng theo độ sâu để thấy cấu trúc cây lời gọi:

In [14]:
def trace_recursive(n, logs):
    ev = [parse(l) for l in logs]
    ans, pos = [0] * n, [0]

    def call(depth):
        i, _, s = ev[pos[0]]
        pos[0] += 1
        pad = "  " * depth
        print(f"{pad}vào hàm {i} (start {s})")
        child = 0
        while ev[pos[0]][1] == "start":
            child += call(depth + 1)
        e = ev[pos[0]][2]
        pos[0] += 1
        total = e - s + 1
        ans[i] += total - child
        print(f"{pad}ra hàm {i} (end {e}): total={total}, con={child}, riêng={total - child}")
        return total

    while pos[0] < len(ev):
        call(0)
    print("=>", ans, "\n")
    return ans


assert trace_recursive(*EX[2][:2]) == [7, 1]

vào hàm 0 (start 0)
  vào hàm 0 (start 2)
  ra hàm 0 (end 5): total=4, con=0, riêng=4
  vào hàm 1 (start 6)
  ra hàm 1 (end 6): total=1, con=0, riêng=1
ra hàm 0 (end 7): total=8, con=5, riêng=3
=> [7, 1] 



## 9. Code nộp LeetCode

Ta nộp **Cách 2** (ngăn xếp + mốc `pre`): ngắn, một lượt, không đệ quy.

In [15]:
class Solution:
    def exclusiveTime(self, n: int, logs: List[str]) -> List[int]:
        stk = []
        ans = [0] * n
        pre = 0
        for log in logs:
            i, op, t = log.split(":")
            i, cur = int(i), int(t)
            if op[0] == "s":
                if stk:
                    ans[stk[-1]] += cur - pre
                stk.append(i)
                pre = cur
            else:
                ans[stk.pop()] += cur - pre + 1
                pre = cur + 1
        return ans


for n, logs, want in EX:
    assert Solution().exclusiveTime(n, logs) == want
print([Solution().exclusiveTime(n, logs) for n, logs, _ in EX])
IMPLS["Solution"] = lambda n, logs: Solution().exclusiveTime(n, logs)

[[3, 4], [8], [7, 1]]


## 10. Vì sao đúng

**Bất biến của Cách 2:** sau khi xử lý mỗi log, mọi ô thời gian **nhỏ hơn `pre`** đã được cộng đúng cho hàm thực sự chạy ở ô đó, và ngăn xếp chứa đúng các lời gọi đang dang dở (đỉnh là hàm sẽ chạy từ ô `pre` trở đi cho tới sự kiện kế tiếp).

- **Khởi tạo:** trước log đầu tiên không ô nào được tính; `pre = 0` không ảnh hưởng vì ngăn xếp rỗng (khi `start` đầu tiên, ta không cộng) và `pre` được gán lại `= cur`.
- **`start` ở `cur`:** hàm đỉnh (nếu có) chạy liên tục các ô `pre .. cur−1` (ô `cur` thuộc về hàm mới vì `start` xảy ra ở **đầu** ô). Cộng `cur − pre` (có thể `0` khi hai sự kiện liên tiếp ở cùng ô). Sau đó `pre = cur`: ô `cur` chưa tính và thuộc về hàm mới ở đỉnh. Bất biến giữ.
- **`end` ở `cur`:** hàm đỉnh (chính là hàm kết thúc, vì lời gọi lồng nhau đúng ngoặc) chạy liên tục các ô `pre .. cur` (kể cả `cur`, vì `end` xảy ra ở **cuối** ô). Cộng `cur − pre + 1`, lấy khỏi ngăn xếp, `pre = cur + 1` (ô tiếp theo chưa tính, do hàm cha — hoặc không ai nếu ngăn xếp rỗng — chạy). Bất biến giữ.
- **Khoảng trống (CPU rảnh):** nếu sau `end` ngăn xếp rỗng rồi tới `start` muộn hơn, `start` không cộng gì (ngăn xếp rỗng) nên các ô rảnh **không** thuộc hàm nào — đúng.
- **Đệ quy:** ngăn xếp lưu các **lời gọi** (có thể trùng id), mỗi lần cộng cho đỉnh nên mỗi ô chỉ cộng đúng **một** lần cho đúng id.

**Cách 3/4:** với mọi lời gọi `c`, các ô của `c` = ô của chính `c` ∪ ô của các con trực tiếp (rời nhau), nên `riêng(c) = tổng(c) − Σ tổng(con)`. Cộng `riêng` của mọi lời gọi cùng id cho `ans[id]`.

**Cách 1:** mô phỏng đúng định nghĩa nên là cơ sở chuẩn ở mức đơn vị. Mục 12 đối chiếu bốn cách với một oracle thứ hai dựng khác hẳn (ghép cặp lời gọi thành khoảng, ô thuộc **khoảng bắt đầu muộn nhất** trong các khoảng chứa nó).

## 11. Phân tích Time / Space Complexity

Gọi `L` là số log (`≤ 500`), `T` là độ dài thời gian (`≤ 10^9 + 1`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng ô | `O(T + L)` | `O(L)` | **không dùng được** khi `T` lớn; chỉ làm oracle |
| 2. Ngăn xếp + `pre` | `O(L)` | `O(L)` | **nộp bài**, một lượt |
| 3. Ngăn xếp khung (tổng − con) | `O(L)` | `O(L)` | dễ mở rộng (độ sâu, thời gian tổng) |
| 4. Đệ quy | `O(L)` | `O(độ sâu) ≤ O(L)` | gọn theo cấu trúc ngoặc; cẩn thận giới hạn đệ quy với `L` rất lớn |

Với `L ≤ 500`, các cách 2–4 chạy trong vài chục micro-giây. Mục 12.6 đo thời gian Cách 1 để thấy nó tuyến tính theo `T`.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (3 ví dụ) cho 4 cách + `Solution`.
2. **Edge case:** một lời gọi, lời gọi 1 đơn vị (`start t`/`end t`), đệ quy chuỗi, nhiều hàm anh em, `n = 100` mà chỉ dùng hàm cuối, timestamp `10^9`, có khoảng trống CPU rảnh, độ sâu 250.
3. **Oracle độc lập** `oracle_intervals`: ghép mỗi `start` với `end` của nó thành **khoảng** `[s, e]`, rồi với mỗi ô, hàm sở hữu ô là khoảng **bắt đầu muộn nhất** trong các khoảng chứa ô (vì lời gọi trong cùng nhất chiếm CPU).
4. **Ngẫu nhiên có seed:** bộ sinh log **hợp lệ** đúng ràng buộc (không hai `start` cùng timestamp, không hai `end` cùng timestamp, ngoặc lồng nhau, có đệ quy và khoảng rảnh).
5. **Timestamp lớn:** kéo giãn mỗi ô thành đoạn dài tới `10^6` (tổng ≤ `10^9`); đáp án kỳ vọng tính từ mô phỏng ô nhỏ có trọng số.
6. **Tính chất:** tổng đáp án = tổng thời gian bận của các lời gọi mức cao nhất; dời mọi timestamp đi một hằng số không đổi đáp án; mọi phần tử `≥ 0`.
7. **Thời gian.**

In [16]:
def oracle_intervals(n, logs):
    """Oracle độc lập: ghép start-end thành khoảng, mỗi ô thuộc khoảng bắt đầu muộn nhất chứa nó."""
    ev = [parse(l) for l in logs]
    open_idx, intervals = [], []                       # intervals: (id, s, e)
    for k, (i, op, t) in enumerate(ev):
        if op == "start":
            open_idx.append(k)
        else:
            k0 = open_idx.pop()
            assert ev[k0][0] == i
            intervals.append((i, ev[k0][2], t))
    assert not open_idx
    ans = [0] * n
    lo = min(s for _, s, _ in intervals)
    hi = max(e for _, _, e in intervals)
    for cell in range(lo, hi + 1):
        best = None
        for i, s, e in intervals:
            if s <= cell <= e and (best is None or s > best[1]):
                best = (i, s)
        if best is not None:
            ans[best[0]] += 1
    return ans


def valid_logs(n, logs):
    """Kiểm tra ràng buộc của đề (và tính hợp lệ lồng ngoặc)."""
    assert 1 <= n <= 100 and 2 <= len(logs) <= 500
    starts, ends, stk = set(), set(), []
    prev = -1
    for log in logs:
        i, op, t = parse(log)
        assert 0 <= i < n and 0 <= t <= 10**9 and t >= prev - 0
        prev = t
        if op == "start":
            assert t not in starts
            starts.add(t); stk.append(i)
        else:
            assert t not in ends
            ends.add(t); assert stk.pop() == i
    assert not stk
    return True


# 1) Ví dụ đề
for n, logs, want in EX:
    assert valid_logs(n, logs)
    assert oracle_intervals(n, logs) == want
    for name, f in IMPLS.items():
        assert f(n, logs) == want, name
print("Ví dụ đề (4 cách + Solution + oracle): OK")

Ví dụ đề (4 cách + Solution + oracle): OK


In [17]:
# 2) Edge case
def depth_logs(d, n=1):
    """Chuỗi d lời gọi lồng nhau (đệ quy), mỗi lớp cách nhau 1 ô, kết thúc ngược lại."""
    logs = [f"{k % n}:start:{k}" for k in range(d)]
    logs += [f"{(d - 1 - k) % n}:end:{d + k}" for k in range(d)]
    return logs


edge = [
    (1, ["0:start:0", "0:end:0"], [1]),                                   # một lời gọi, một ô
    (1, ["0:start:5", "0:end:9"], [5]),                                   # bắt đầu muộn
    (2, ["0:start:0", "1:start:1", "1:end:1", "0:end:2"], [2, 1]),                  # con chạy đúng 1 ô
    (3, ["0:start:0", "1:start:1", "2:start:2", "2:end:3", "1:end:4", "0:end:5"], [2, 2, 2]),
    (2, ["0:start:0", "0:end:2", "1:start:3", "1:end:5"], [3, 3]),        # hai lời gọi liền nhau (anh em mức cao nhất)
    (2, ["0:start:0", "0:end:2", "1:start:10", "1:end:11"], [3, 2]),      # khoảng rảnh CPU từ ô 3..9
    (1, ["0:start:0", "0:start:1", "0:end:1", "0:start:2", "0:end:2", "0:end:3"], [4]),   # hai con liền nhau
    (100, ["99:start:0", "99:end:4"], [0] * 99 + [5]),                    # chỉ dùng hàm cuối
    (2, ["0:start:0", "1:start:999999999", "1:end:999999999", "0:end:1000000000"], [10**9, 1]),   # timestamp 10^9
    (1, depth_logs(250), [500]),                                          # độ sâu 250 (500 log)
]
for n, logs, want in edge:
    assert valid_logs(n, logs), logs[:3]
    for name, f in IMPLS.items():
        if name == "unit" and max(parse(l)[2] for l in logs) > 10**5:
            continue                                                       # mô phỏng ô quá chậm với timestamp lớn
        assert f(n, logs) == want, (name, logs[:4], f(n, logs), want)
    if max(parse(l)[2] for l in logs) <= 600:
        assert oracle_intervals(n, logs) == want
print(f"Edge case: OK ({len(edge)} test, gồm timestamp 10^9 và độ sâu 250)")

Edge case: OK (10 test, gồm timestamp 10^9 và độ sâu 250)


In [18]:
# 3-4) Bộ sinh log hợp lệ ngẫu nhiên (ở "mức ô"): tối đa một start và một end mỗi ô, ngoặc lồng nhau đúng
def gen_unit_events(rng, n, max_calls, p_start=0.5, p_end=0.5, p_gap=0.3):
    ev, stk, calls, u = [], [], 0, 0                  # ev: (id, op, ô)
    while True:
        if not stk:
            if calls >= max_calls:
                break
            if ev and rng.random() < p_gap:           # khoảng rảnh CPU
                u += rng.randint(1, 3)
        # đầu ô u: có thể start một lời gọi
        if calls < max_calls and (not stk or rng.random() < p_start):
            f = rng.randrange(n)
            ev.append((f, "start", u)); stk.append(f); calls += 1
        # ô u được chạy bởi đỉnh; cuối ô u: có thể end
        if stk and (calls >= max_calls or rng.random() < p_end):
            ev.append((stk.pop(), "end", u))
        u += 1
    return ev


def to_logs(ev):
    return [f"{i}:{op}:{t}" for i, op, t in ev]


rng = random.Random(636)
tests = []
for _ in range(3000):
    n = rng.randint(1, 5)
    calls = rng.randint(1, 40)
    ev = gen_unit_events(rng, n, calls, rng.choice([0.3, 0.5, 0.8]), rng.choice([0.2, 0.5, 0.8]))
    logs = to_logs(ev)
    tests.append((n, logs))
    assert valid_logs(n, logs)

depths = set()
for n, logs in tests:
    want = oracle_intervals(n, logs)
    for name, f in IMPLS.items():
        assert f(n, logs) == want, (name, n, logs, f(n, logs), want)
    # đo độ sâu lớn nhất để chắc có đệ quy/lồng sâu
    d = m = 0
    for l in logs:
        d += 1 if parse(l)[1] == "start" else -1
        m = max(m, d)
    depths.add(m)
print(f"Ngẫu nhiên: {len(tests)} test x 5 cài đặt trùng oracle; độ sâu lồng nhau đã gặp: {sorted(depths)[:3]} ... {max(depths)}")
assert max(depths) >= 8
assert any(len({parse(l)[0] for l in logs}) == 1 and len(logs) > 6 for n, logs in tests)   # có test đệ quy thuần

Ngẫu nhiên: 3000 test x 5 cài đặt trùng oracle; độ sâu lồng nhau đã gặp: [1, 2, 3] ... 36


In [19]:
# 5) Timestamp lớn: kéo giãn mỗi ô thành đoạn dài; đáp án = tổng độ dài các ô được hàm đó chiếm
def stretch(ev, rng, maxlen):
    last = max(t for _, _, t in ev)
    lens = [rng.randint(1, maxlen) for _ in range(last + 2)]
    pre = [0]
    for L in lens:
        pre.append(pre[-1] + L)
    logs = []
    for i, op, u in ev:
        logs.append(f"{i}:{op}:{pre[u] if op == 'start' else pre[u + 1] - 1}")
    return logs, lens


def weighted_expected(n, ev, lens):
    """Mô phỏng ô nhỏ (oracle_units) rồi nhân trọng số độ dài từng ô."""
    ans = [0] * n
    stk, p, u = [], 0, ev[0][2]
    last = max(t for _, _, t in ev)
    while u <= last:
        while p < len(ev) and ev[p][1] == "start" and ev[p][2] == u:
            stk.append(ev[p][0]); p += 1
        if stk:
            ans[stk[-1]] += lens[u]
        while p < len(ev) and ev[p][1] == "end" and ev[p][2] == u:
            stk.pop(); p += 1
        u += 1
    return ans


rng = random.Random(1000000000)
mx = 0
for _ in range(1000):
    n = rng.randint(1, 6)
    ev = gen_unit_events(rng, n, rng.randint(1, 120))
    logs, lens = stretch(ev, rng, 10**6)
    assert valid_logs(n, logs)
    want = weighted_expected(n, ev, lens)
    for name in ("stack", "frames", "recursive", "Solution"):
        assert IMPLS[name](n, logs) == want, name
    mx = max(mx, max(parse(l)[2] for l in logs))
print("Timestamp lớn: 1000 test x 4 cách trùng mô phỏng có trọng số; timestamp lớn nhất đã gặp:", mx)
assert mx > 10**7

Timestamp lớn: 1000 test x 4 cách trùng mô phỏng có trọng số; timestamp lớn nhất đã gặp: 132632763


In [20]:
# 6) Tính chất: tổng = thời gian bận mức cao nhất; bất biến khi dời mốc thời gian; không âm
def busy_time(logs):
    depth, total, s0 = 0, 0, None
    for l in logs:
        i, op, t = parse(l)
        if op == "start":
            if depth == 0:
                s0 = t
            depth += 1
        else:
            depth -= 1
            if depth == 0:
                total += t - s0 + 1
    return total


def shift(logs, d):
    return [f"{i}:{op}:{t + d}" for i, op, t in map(parse, logs)]


rng = random.Random(66)
for _ in range(500):
    n = rng.randint(1, 6)
    ev = gen_unit_events(rng, n, rng.randint(1, 60))
    logs, _ = stretch(ev, rng, 1000)
    res = exclusive_stack(n, logs)
    assert all(x >= 0 for x in res)
    assert sum(res) == busy_time(logs)
    d = rng.randint(0, 10**9 - max(parse(l)[2] for l in logs))
    assert exclusive_stack(n, shift(logs, d)) == res
print("Tính chất: tổng = thời gian bận, bất biến khi dời mốc, không âm: OK")

Tính chất: tổng = thời gian bận, bất biến khi dời mốc, không âm: OK


In [21]:
# 7) Thời gian: Cách 1 tuyến tính theo T (độ dài thời gian), Cách 2 không phụ thuộc T
def make_big(scale):
    # hàm 0 bao hàm hàm 1; kéo giãn bằng scale
    return 2, ["0:start:0", f"1:start:{scale}", f"1:end:{3 * scale}", f"0:end:{4 * scale}"]


for scale in (10**3, 10**4, 10**5, 10**6):
    n, logs = make_big(scale)
    t0 = time.perf_counter(); a = exclusive_unit(n, logs) if scale <= 10**5 else None; d1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); b = exclusive_stack(n, logs); d2 = (time.perf_counter() - t0) * 1000
    assert b == [2 * scale, 2 * scale + 1]
    if a is not None:
        assert a == b
    print(f"T≈{4 * scale:>8}: unit {('%9.2f ms' % d1) if a is not None else '   (bỏ qua)  '} | stack {d2:.4f} ms | kết quả {b}")

T≈    4000: unit      0.69 ms | stack 0.0052 ms | kết quả [2000, 2001]
T≈   40000: unit      7.32 ms | stack 0.0075 ms | kết quả [20000, 20001]


T≈  400000: unit     72.84 ms | stack 0.0136 ms | kết quả [200000, 200001]
T≈ 4000000: unit    (bỏ qua)   | stack 0.0050 ms | kết quả [2000000, 2000001]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `+ 1` khi gặp `end`

`end` xảy ra ở **cuối** ô `cur`, nên ô `cur` vẫn thuộc hàm vừa kết thúc: độ dài là `cur − pre + 1`. Quên `+ 1` làm thiếu đúng 1 đơn vị cho **mỗi** lần `end`.

In [22]:
def wrong_no_plus_one(n, logs):
    stk, ans, pre = [], [0] * n, 0
    for log in logs:
        i, op, cur = parse(log)
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
        else:
            ans[stk.pop()] += cur - pre              # sai: thiếu + 1
            pre = cur + 1
    return ans


n, logs, want = EX[0]
print("sai :", wrong_no_plus_one(n, logs), "| đúng:", exclusive_stack(n, logs))
assert wrong_no_plus_one(n, logs) == [2, 3] and exclusive_stack(n, logs) == want

sai : [2, 3] | đúng: [3, 4]


### Lỗi 2: đặt `pre = cur` (thay vì `cur + 1`) sau `end`

Sau `end ở cur`, hàm cha chỉ chạy tiếp từ ô `cur + 1`. Nếu đặt `pre = cur`, ô `cur` bị **tính hai lần** (một lần cho hàm con, một lần cho hàm cha).

In [23]:
def wrong_pre_cur(n, logs):
    stk, ans, pre = [], [0] * n, 0
    for log in logs:
        i, op, cur = parse(log)
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
        else:
            ans[stk.pop()] += cur - pre + 1
            pre = cur                                 # sai: phải là cur + 1
    return ans


print("sai :", wrong_pre_cur(*EX[0][:2]), "| đúng:", exclusive_stack(*EX[0][:2]))
assert wrong_pre_cur(*EX[0][:2]) == [4, 4]

sai : [4, 4] | đúng: [3, 4]


### Lỗi 3: quên cộng thời gian cho **hàm cha** khi hàm con bắt đầu

Nhánh `start` phải cộng `cur − pre` cho hàm đỉnh **trước khi** đẩy hàm mới; nếu bỏ, khoảng thời gian hàm cha đã chạy trước khi gọi con bị mất.

In [24]:
def wrong_no_parent_credit(n, logs):
    stk, ans, pre = [], [0] * n, 0
    for log in logs:
        i, op, cur = parse(log)
        if op == "start":
            stk.append(i)                             # sai: thiếu ans[stk[-1]] += cur - pre
            pre = cur
        else:
            ans[stk.pop()] += cur - pre + 1
            pre = cur + 1
    return ans


print("sai :", wrong_no_parent_credit(*EX[0][:2]), "| đúng:", exclusive_stack(*EX[0][:2]))
assert wrong_no_parent_credit(*EX[0][:2]) == [1, 4]

sai : [1, 4] | đúng: [3, 4]


### Lỗi 4: không đổi timestamp sang `int`

`split(":")` cho **chuỗi**. Trừ hai chuỗi gây `TypeError`; và nếu chỉ so sánh/ghép chuỗi sẽ cho kết quả sai âm thầm (`"10" < "9"`).

In [25]:
def wrong_str_time(n, logs):
    stk, ans, pre = [], [0] * n, "0"
    for log in logs:
        i, op, cur = log.split(":")                   # cur là chuỗi!
        i = int(i)
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
        else:
            ans[stk.pop()] += cur - pre + 1
            pre = cur + 1
    return ans


try:
    wrong_str_time(*EX[0][:2])
except TypeError as e:
    print("TypeError:", e)
print('"10" < "9" ->', "10" < "9", "(so chuỗi theo từ điển, không phải theo số)")
assert exclusive_stack(*EX[0][:2]) == [3, 4]

TypeError: unsupported operand type(s) for -: 'str' and 'str'
"10" < "9" -> True (so chuỗi theo từ điển, không phải theo số)


### Lỗi 5: cộng **thời gian tổng** mà không trừ thời gian hàm con (đếm trùng khi đệ quy)

Nếu mỗi lời gọi cộng `end − start + 1` thẳng vào `ans[id]`, thời gian của hàm con được tính **cả cho con lẫn cho cha** (và với đệ quy, cùng một hàm bị cộng nhiều lần cho cùng khoảng).

In [26]:
def wrong_inclusive(n, logs):
    stk, ans = [], [0] * n
    for log in logs:
        i, op, t = parse(log)
        if op == "start":
            stk.append((i, t))
        else:
            fid, s = stk.pop()
            ans[fid] += t - s + 1                     # sai: chưa trừ thời gian các con
    return ans


print("sai :", wrong_inclusive(*EX[1][:2]), "(Ví dụ 2) | đúng:", exclusive_stack(*EX[1][:2]))
print("sai :", wrong_inclusive(*EX[0][:2]), "(Ví dụ 1) | đúng:", exclusive_stack(*EX[0][:2]))
assert wrong_inclusive(*EX[1][:2]) == [13] and wrong_inclusive(*EX[0][:2]) == [7, 4]

sai : [13] (Ví dụ 2) | đúng: [8]
sai : [7, 4] (Ví dụ 1) | đúng: [3, 4]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** dữ liệu là chuỗi sự kiện **mở/đóng lồng nhau** (gọi hàm, thẻ HTML, ngoặc, công việc con) ⇒ **ngăn xếp**. Câu hỏi "ai đang chạy?" ⇔ "đỉnh ngăn xếp là gì?".
- **Tính theo chênh lệch giữa hai sự kiện liên tiếp** thay vì đi từng đơn vị thời gian; đây là kỹ thuật **sweep line / event simulation**.
- **Ghi nhớ hai quy ước mốc:** `start` ở **đầu** ô (`cur − pre`, mốc mới `pre = cur`), `end` ở **cuối** ô (`cur − pre + 1`, mốc mới `pre = cur + 1`). Khi nghi ngờ, **vẽ dòng thời gian các ô** (mục 4) và thử với `start t / end t` (một ô).
- **Khoảng đóng `[s, e]` dài `e − s + 1`.** Lỗi lệch 1 là lỗi số một ở bài này.
- **Đệ quy cùng id:** ngăn xếp lưu **lời gọi**, không phải tập hàm; đừng dùng `set` hay mảng "đang chạy" theo id.
- **Hai cách nhìn tương đương:** "mốc `pre`" (Cách 2) và "tổng − con" (Cách 3). Nếu đề đòi thêm **thời gian tổng (inclusive)** hay **độ sâu**, Cách 3 mở rộng tự nhiên.
- **Cùng họ:** *Valid Parentheses* (20), *Decode String* (394), *Basic Calculator* (224), *Daily Temperatures* (739), *Remove Outermost Parentheses* (1021), *Validate Stack Sequences* (946).
- **Tự test:** viết một **oracle ngây thơ** (từng ô) và bộ sinh log hợp lệ ngẫu nhiên — mục 12.

## 15. Biến thể của bài

1. **Thời gian bận / rảnh của CPU:** `busy = Σ ans`; `idle = (timestamp end cuối − timestamp start đầu + 1) − busy`.
2. **Độ sâu lồng nhau tối đa** (độ sâu ngăn xếp lớn nhất).
3. **Thời gian tổng (inclusive) của từng hàm** (đệ quy chỉ tính một lần): bài tập 5.
4. **Log không theo thứ tự thời gian:** sắp xếp theo `(timestamp, start trước end)` rồi chạy như cũ — cẩn thận với thứ tự khi `start` và `end` cùng timestamp.
5. **Nhiều CPU/đa luồng:** mỗi luồng một ngăn xếp riêng.
6. **Log có thể thiếu `end`** (chương trình bị cắt): đóng các lời gọi còn dang dở tại thời điểm cuối.

Cài nhanh biến thể 1 + 2 và kiểm tra bằng oracle:

In [27]:
def profile(n, logs):
    """Trả về (exclusive, idle_time, max_depth)."""
    stk, ans, pre = [], [0] * n, 0
    depth_max = 0
    first = last = None
    for log in logs:
        i, op, cur = parse(log)
        if first is None:
            first = cur
        last = cur
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
            depth_max = max(depth_max, len(stk))
        else:
            ans[stk.pop()] += cur - pre + 1
            pre = cur + 1
    return ans, (last - first + 1) - sum(ans), depth_max


assert profile(1, ["0:start:0", "0:end:2", "0:start:10", "0:end:11"]) == ([5], 7, 1)
assert profile(*EX[1][:2]) == ([8], 0, 2)


def unit_profile(n, logs):
    """Oracle ở mức ô: đếm ô rảnh và độ sâu bằng mô phỏng."""
    ev = [parse(l) for l in logs]
    stk, p, idle, depth = [], 0, 0, 0
    t, last = ev[0][2], max(e[2] for e in ev)
    while t <= last:
        while p < len(ev) and ev[p][1] == "start" and ev[p][2] == t:
            stk.append(ev[p][0]); depth = max(depth, len(stk)); p += 1
        if not stk:
            idle += 1
        while p < len(ev) and ev[p][1] == "end" and ev[p][2] == t:
            stk.pop(); p += 1
        t += 1
    return idle, depth


for n, logs in tests[:1500]:
    ans, idle, depth = profile(n, logs)
    assert ans == oracle_intervals(n, logs) and (idle, depth) == unit_profile(n, logs)
print("Biến thể (idle + độ sâu) khớp oracle trên 1500 test: OK;  ví dụ:", profile(*EX[2][:2]))

Biến thể (idle + độ sâu) khớp oracle trên 1500 test: OK;  ví dụ:

 ([7, 1], 0, 2)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng ô | cộng 1 cho đỉnh ở mỗi ô | `O(T + L)` | `O(L)` | chỉ làm **oracle**; `T` nhỏ |
| 2. Ngăn xếp + `pre` | cộng độ dài đoạn giữa hai sự kiện cho đỉnh | `O(L)` | `O(L)` | **làm bài**: ngắn, một lượt |
| 3. Khung gọi: tổng − con | mỗi lời gọi tính tổng rồi trừ phần con | `O(L)` | `O(L)` | cần thêm số liệu từng lời gọi (độ sâu, tổng) |
| 4. Đệ quy | cấu trúc ngoặc → cây lời gọi | `O(L)` | `O(độ sâu)` | log nhỏ, muốn code theo cấu trúc; cẩn thận giới hạn đệ quy |

Trực giác chọn: **ngăn xếp + mốc `pre`** là mặc định; chuyển sang **khung gọi** khi bài đòi thêm thông tin trên từng lời gọi.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

```python
print(exclusive_stack(1, ["0:start:0", "0:end:3"]))
print(exclusive_stack(2, ["0:start:0", "1:start:1", "1:end:1", "0:end:2"]))
```

In [28]:
print(exclusive_stack(1, ["0:start:0", "0:end:3"]))
print(exclusive_stack(2, ["0:start:0", "1:start:1", "1:end:1", "0:end:2"]))

[4]
[2, 1]


### Bài 2 (tính tay)

`n = 2`, `logs = ["0:start:0","1:start:1","1:end:2","1:start:4","1:end:4","0:end:6"]`. Vẽ dòng thời gian các ô `0..6`, đếm ô của từng hàm, và ghi vào `my_answer_2` (list hai số; để `None` đến khi tính xong).

In [29]:
logs_b2 = ["0:start:0", "1:start:1", "1:end:2", "1:start:4", "1:end:4", "0:end:6"]
my_answer_2 = None          # TODO: điền [thời gian hàm 0, thời gian hàm 1] tính tay

### Bài 3 (điền TODO)

Hoàn thành hàm `exclusive_todo` theo Cách 2: bạn cần viết nhánh `end` (cộng cho hàm vừa kết thúc, cập nhật `pre`). Cell chỉ **định nghĩa**, không gọi.

In [30]:
def exclusive_todo(n, logs):
    stk, ans, pre = [], [0] * n, 0
    for log in logs:
        i, op, t = log.split(":")
        i, cur = int(i), int(t)
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
        else:
            # TODO: cộng cho hàm ở đỉnh (lấy nó ra khỏi ngăn xếp) và cập nhật pre
            pass
    return ans

### Bài 4 (tự viết lại)

Tự viết `exclusive_mine(n, logs)` theo **Cách 3** (khung gọi `[id, start, child]`, tổng − con) mà không nhìn code cũ. Cell dưới chỉ để khung rỗng.

In [31]:
def exclusive_mine(n, logs):
    # TODO: viết theo Cách 3
    pass

### Bài 5 (biến thể / mini challenge)

Viết `inclusive_once(n, logs)` trả về, cho mỗi hàm, **số ô thời gian trong đó hàm xuất hiện trong ngăn xếp ít nhất một lần** (thời gian tổng, đệ quy chỉ tính **một lần**). Ví dụ 1 cho `[7, 4]` (hàm 0 có mặt các ô `0..6`, hàm 1 các ô `2..5`); Ví dụ 2 cho `[8]`.

Gợi ý: giữ `cnt[i]` = số lời gọi đang dang dở của hàm `i` và `since[i]` = ô bắt đầu của lời gọi **ngoài cùng**; khi `cnt` về `0` thì cộng khoảng.

In [32]:
def inclusive_once(n, logs):
    # TODO: viết hàm
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: vẽ các ô; hàm nào chiếm ô nào. `start 0, end 3` là bốn ô.
- Bài 2: liệt kê ô `0..6` và ai chiếm từng ô.
- Bài 3: khi `end`: hàm ở đỉnh chạy từ `pre` đến `cur` (gồm cả `cur`).
- Bài 4: mỗi khung có ba phần: id, start, tổng thời gian của các con.
- Bài 5: cần biết khi nào lời gọi **ngoài cùng** của hàm bắt đầu/kết thúc.

**Mức 2 (vừa):**
- Bài 1: ví dụ thứ hai: hàm 1 chỉ chạy ô 1; hàm 0 chạy ô 0 và ô 2.
- Bài 2: hàm 1 chiếm ô `1,2` và ô `4`; hàm 0 chiếm các ô còn lại.
- Bài 3: `ans[stk.pop()] += cur - pre + 1`, rồi `pre = cur + 1`.
- Bài 4: khi `end`: `total = t - s + 1`; `ans[id] += total - child`; nếu còn khung cha thì `child_cha += total`.
- Bài 5: `start`: `cnt[i] += 1`, nếu `cnt[i] == 1` thì `since[i] = t`. `end`: `cnt[i] -= 1`, nếu `cnt[i] == 0` thì `res[i] += t - since[i] + 1`.

**Mức 3 (gần đáp án):**
- Bài 1: `[4]` và `[2, 1]`.
- Bài 2: hàm 0: ô `0, 3, 5, 6` → 4; hàm 1: ô `1, 2, 4` → 3 ⇒ `[4, 3]`.
- Bài 3: đúng hai dòng như mức 2.
- Bài 4: nhớ `stk.pop()` trả về khung ba phần để gán `fid, s, child = stk.pop()`.
- Bài 5: kiểm tra với Ví dụ 1 → `[7, 4]`, Ví dụ 2 → `[8]` (hàm 0 đệ quy nhưng chỉ tính lời gọi ngoài cùng: ô `0..7`).

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [33]:
# Bài 1
assert exclusive_stack(1, ["0:start:0", "0:end:3"]) == [4]
assert exclusive_stack(2, ["0:start:0", "1:start:1", "1:end:1", "0:end:2"]) == [2, 1]
print("Bài 1: [4] và [2, 1]")

# Bài 2
ans_2 = [4, 3]
assert exclusive_stack(2, logs_b2) == ans_2 == oracle_intervals(2, logs_b2)
print("Bài 2:", ans_2, "(hàm 0: ô 0,3,5,6; hàm 1: ô 1,2,4)")


# Bài 3
def exclusive_todo_sol(n, logs):
    stk, ans, pre = [], [0] * n, 0
    for log in logs:
        i, op, t = log.split(":")
        i, cur = int(i), int(t)
        if op == "start":
            if stk:
                ans[stk[-1]] += cur - pre
            stk.append(i)
            pre = cur
        else:
            ans[stk.pop()] += cur - pre + 1
            pre = cur + 1
    return ans


for n, logs, want in EX:
    assert exclusive_todo_sol(n, logs) == want
for n, logs in tests[:300]:
    assert exclusive_todo_sol(n, logs) == exclusive_stack(n, logs)
print("Bài 3: OK")

Bài 1: [4] và [2, 1]
Bài 2: [4, 3] (hàm 0: ô 0,3,5,6; hàm 1: ô 1,2,4)
Bài 3: OK


In [34]:
# Bài 4
def exclusive_mine_sol(n, logs):
    stk, ans = [], [0] * n
    for log in logs:
        i, op, t = parse(log)
        if op == "start":
            stk.append([i, t, 0])
        else:
            fid, s, child = stk.pop()
            total = t - s + 1
            ans[fid] += total - child
            if stk:
                stk[-1][2] += total
    return ans


for n, logs, want in EX:
    assert exclusive_mine_sol(n, logs) == want
for n, logs in tests[:500]:
    assert exclusive_mine_sol(n, logs) == oracle_intervals(n, logs)
print("Bài 4: OK")


# Bài 5
def inclusive_once_sol(n, logs):
    cnt, since, res = [0] * n, [0] * n, [0] * n
    for log in logs:
        i, op, t = parse(log)
        if op == "start":
            cnt[i] += 1
            if cnt[i] == 1:
                since[i] = t
        else:
            cnt[i] -= 1
            if cnt[i] == 0:
                res[i] += t - since[i] + 1
    return res


def inclusive_oracle(n, logs):
    ev = [parse(l) for l in logs]
    stk, p, t = [], 0, ev[0][2]
    res, last = [0] * n, max(e[2] for e in ev)
    while t <= last:
        while p < len(ev) and ev[p][1] == "start" and ev[p][2] == t:
            stk.append(ev[p][0]); p += 1
        for f in set(stk):
            res[f] += 1
        while p < len(ev) and ev[p][1] == "end" and ev[p][2] == t:
            stk.pop(); p += 1
        t += 1
    return res


assert inclusive_once_sol(*EX[0][:2]) == [7, 4] and inclusive_once_sol(*EX[1][:2]) == [8]
for n, logs in tests[:1000]:
    assert inclusive_once_sol(n, logs) == inclusive_oracle(n, logs)
print("Bài 5:", inclusive_once_sol(*EX[0][:2]), inclusive_once_sol(*EX[1][:2]), inclusive_once_sol(*EX[2][:2]))

Bài 4: OK
Bài 5: [7, 4] [8] [8, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (ngăn xếp + mốc `pre`):**

```python
class Solution:
    def exclusiveTime(self, n, logs):
        stk, ans, pre = [], [0] * n, 0
        for log in logs:
            i, op, t = log.split(":")
            i, cur = int(i), int(t)
            if op[0] == "s":                      # start: ở ĐẦU ô cur
                if stk:
                    ans[stk[-1]] += cur - pre     # cha chạy [pre, cur-1]
                stk.append(i)
                pre = cur
            else:                                 # end: ở CUỐI ô cur
                ans[stk.pop()] += cur - pre + 1   # chạy [pre, cur]
                pre = cur + 1
        return ans
```

**Công thức nhớ:** *start: `cur − pre`, `pre = cur`; end: `cur − pre + 1`, `pre = cur + 1`.*

**Checklist trước khi submit:**

- [ ] Đã đổi `id` và `timestamp` sang `int`.
- [ ] `start`: cộng `cur - pre` cho **hàm đỉnh (cha)** nếu ngăn xếp không rỗng, **rồi** mới đẩy hàm mới; `pre = cur`.
- [ ] `end`: cộng `cur - pre + 1` cho hàm **bị lấy ra**; `pre = cur + 1` (không phải `cur`).
- [ ] Ngăn xếp chứa **lời gọi** (có thể trùng id khi đệ quy), không phải tập id.
- [ ] Không mô phỏng từng ô (timestamp tới `10^9`).
- [ ] Đã thử: một lời gọi một ô (`start t`, `end t`), đệ quy, hai anh em liền nhau, khoảng rảnh CPU, timestamp `10^9`.
- [ ] Đã so với oracle mô phỏng ô nhỏ trên log ngẫu nhiên hợp lệ.